# Prueba 1: ¿hasta dónde llega el efecto electrónico de la glicina en GluN1?

Geometría congelada (1PB7 con hidrógenos, preparada en local). PM7, MOZYME, `1SCF GRADIENTS`.

| Sistema | Contenido |
|---|---|
| A | proteína + glicina |
| B | proteína sola |
| C | proteína + la glicina como cargas puntuales (sus cargas ESP de PM7, `MM_CHARGES`) |
| D | glicina sola (cargas ESP) |

A − B es todo el efecto de la glicina sobre las cargas y fuerzas de la proteína; C − B, lo que da un campo clásico; A − C,
lo que la glicina cuántica añade. Variantes: dos corridas iguales (ruido), CPU (`MOPAC_NOGPU=1`), `CUTOFF=15` y `25`
(MOZYME trata las interacciones más allá de ~10 Å como cargas y dipolos), agua implícita (`EPS=78.4`) y SCF convencional
sin orbitales localizados.

| Bloque | Tiempo (A100, estimado) |
|---|---|
| Compilar MOPAC | ~10 min |
| GPU: base, repeticiones, cutoffs, agua | ~10–20 min |
| CPU: A, B, C en paralelo | ~15–30 min (corre a la vez que la GPU) |
| SCF convencional, 4 614 átomos | hasta 45 min por corrida (límite) |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*, y luego *Ejecutar todo*. Los resultados
se guardan en Google Drive (`MyDrive/mopac_paper_bench/RESULTS_NAME`); si la sesión se corta, vuelva a ejecutar todo y lo
terminado se salta. La última celda descarga un zip.

## 1. Configuración

In [ ]:
COMMIT = '323b6076'
RESULTS_NAME = 'allo_t1_a100'
USE_DRIVE = True
REQUIRE_GPU = 'A100'           # None: aceptar cualquier GPU
CONV_TIMEOUT = 2700            # s por corrida del SCF convencional

## 2. GPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json, gzip, base64, threading
from pathlib import Path

def run(cmd, check=True, env=None):
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
NCPU = os.cpu_count()
CC = gpu.split(',')[-1].strip().replace('.', '')
if REQUIRE_GPU and REQUIRE_GPU not in gpu:
    raise RuntimeError(f'Esta sesión tiene {gpu}, no una {REQUIRE_GPU}.')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/mopac_paper_bench')
else:
    BASE = Path('/content/mopac_paper_bench')
OUT = BASE / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'ncpu': NCPU, 'commit': COMMIT}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
MOPAC = BUILD / 'mopac'
if not MOPAC.exists():
    run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
        'libopenblas-dev liblapack-dev > /dev/null')
    os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
    shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
    run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
    run(f'git -C {SRC} checkout -q {COMMIT}')
    head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
    if not head.startswith(COMMIT):
        raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
    stamp = OUT / 'commit.txt'
    if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
        raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
    stamp.write_text(head + '\n')
    t0 = time.time()
    run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
        f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
    run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
        f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
    print(f'compilado en {time.time() - t0:.0f} s')
run(f'ls -la {MOPAC}')

## 4. Entradas (estructura preparada en local y scripts)

In [ ]:
Path('/content/t1_tools.py').write_text('"""Test 1: how far the glycine changes the protein\'s electrons, at a frozen geometry (MOZYME, PM7).\n\n    python t1_tools.py decks ATOMS_CSV WORK_DIR CHARGE      write the decks (A, B, D; C after D)\n    python t1_tools.py mmcharges WORK_DIR                   C\'s point charges from D\'s ESP charges\n    python t1_tools.py analyse WORK_DIR OUT_DIR             tables and figures\n\nA = protein + glycine, B = protein alone, C = protein + the glycine as PM7 ESP point charges (MM_CHARGES),\nD = glycine alone (ESP charges). Every protein atom keeps its place and its index in A, B and C, so\nA - B is the whole effect of the glycine on the protein\'s charges and forces, C - B the part a fixed set of\npoint charges gives (the classical field), A - C what the quantum glycine adds (charge transfer, exchange,\npolarisation of the glycine, its true charge distribution). Repeats of the same job give the noise.\n"""\nimport csv\nimport json\nimport re\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nCOMMON = "PM7 1SCF GRADIENTS GEO-OK AUX(PRECISION=4) SCFCRT=1.D-7 ITRY=500"\n\n\ndef load(atoms_csv):\n    rows = list(csv.DictReader(open(atoms_csv)))\n    prot = [(r["element"], np.array([float(r[k]) for k in "xyz"])) for r in rows if r["resname"] != "LIG"]\n    lig = [(r["element"], np.array([float(r[k]) for k in "xyz"])) for r in rows if r["resname"] == "LIG"]\n    meta = [(r["resname"], r["resid"], r["atom"], r["kind"]) for r in rows if r["resname"] != "LIG"]\n    return prot, lig, meta\n\n\ndef deck(atoms, kw, title):\n    lines = [kw, title, ""]\n    lines += [f"{el:2s} {x[0]:12.6f} 1 {x[1]:12.6f} 1 {x[2]:12.6f} 1" for el, x in atoms]\n    return "\\n".join(lines) + "\\n"\n\n\nVARIANTS = {          # name: (extra keywords, environment, MOZYME?)\n    "vac": ("", {}, True), "r2": ("", {}, True), "cpu": ("", {"MOPAC_NOGPU": "1"}, True),\n    "c15": ("CUTOFF=15", {}, True), "c25": ("CUTOFF=25", {}, True), "eps": ("EPS=78.4", {}, True),\n    "conv": ("", {}, False),\n}\n\n\ndef write_decks(atoms_csv, wd, charge):\n    prot, lig, meta = load(atoms_csv)\n    wd.mkdir(parents=True, exist_ok=True)\n    (wd / "meta.json").write_text(json.dumps({"meta": meta, "lig": [x.tolist() for _, x in lig],\n                                              "prot": [x.tolist() for _, x in prot]}))\n    (wd / "D").mkdir(exist_ok=True)\n    (wd / "D" / "job.mop").write_text(deck(lig, "PM7 1SCF CHARGE=0 ESP GEO-OK AUX(PRECISION=4)", "glycine alone"))\n    for v, (kw, env, mozyme) in VARIANTS.items():\n        for sysname, atoms in (("A", prot + lig), ("B", prot), ("C", prot)):\n            d = wd / f"{sysname}_{v}"\n            d.mkdir(exist_ok=True)\n            extra = (" MOZYME" if mozyme else "") + (f" {kw}" if kw else "")\n            if sysname == "C":\n                extra += \' MM_CHARGES="glycine.chg"\' if mozyme else " QMMM"\n            (d / "job.mop").write_text(deck(atoms, f"{COMMON} CHARGE={charge}{extra}", f"{sysname} {v}"))\n            (d / "env.json").write_text(json.dumps(env))\n\n\ndef aux_block(text, key):\n    m = re.search(r"(?<![A-Z_])" + re.escape(key) + r"(?::[A-Z/]+)?\\[(\\d+)\\]=", text)\n    if not m:\n        return None\n    n, vals = int(m.group(1)), []\n    for ln in text[m.end():].splitlines():\n        if ln.strip().startswith("#"):\n            continue\n        for tok in ln.split():\n            vals.append(float(tok.replace("D", "E")))\n            if len(vals) == n:\n                return np.array(vals)\n    return None\n\n\ndef write_mmcharges(wd):\n    meta = json.loads((wd / "meta.json").read_text())\n    q = aux_block((wd / "D" / "job.aux").read_text(errors="ignore"), "ELECTROSTATIC_POTENTIAL_CHARGES")\n    lig = np.array(meta["lig"])\n    prot = np.array(meta["prot"])\n    text = "".join(f"{x[0]:.6f} {x[1]:.6f} {x[2]:.6f} {c:.6f}\\n" for x, c in zip(lig, q))\n    pot = [332.0637 * float((q / np.linalg.norm(lig - x, axis=1)).sum()) for x in prot]\n    molin = f"glycine ESP charges, potential at each protein atom (kcal/mol, +1 charge)\\n{len(prot)} 0\\n" + \\\n        "".join(f"x y z q {v:.6f}\\n" for v in pot)\n    for d in wd.glob("C_*"):\n        (d / "glycine.chg").write_text(text)\n        (d / "mol.in").write_text(molin)\n    return q\n\n\ndef read_run(d, n):\n    aux = d / "job.aux"\n    if not aux.exists():\n        return None\n    t = aux.read_text(errors="ignore")\n    q = aux_block(t, "ATOM_CHARGES")\n    g = aux_block(t, "GRADIENTS")\n    x = aux_block(t, "ATOM_X_OPT") if aux_block(t, "ATOM_X_OPT") is not None else aux_block(t, "ATOM_X")\n    if q is None:\n        return None\n    return {"q": q[:n], "g": g.reshape(-1, 3)[:n] if g is not None and len(g) >= 3 * n else None,\n            "x": x.reshape(-1, 3)[:n] if x is not None else None}\n\n\nBINS = [0, 4, 6, 8, 10, 12, 15, 20, 25, 30, 40]\n\n\ndef binned(r, v):\n    out = []\n    for lo, hi in zip(BINS, BINS[1:]):\n        s = v[(r >= lo) & (r < hi)]\n        out.append((lo, hi, len(s), np.median(s) if len(s) else np.nan, np.percentile(s, 95) if len(s) else np.nan,\n                    s.max() if len(s) else np.nan))\n    return out\n\n\ndef analyse(wd, od):\n    od.mkdir(parents=True, exist_ok=True)\n    meta = json.loads((wd / "meta.json").read_text())\n    prot, lig = np.array(meta["prot"]), np.array(meta["lig"])\n    n = len(prot)\n    r = np.linalg.norm(prot[:, None] - lig[None], axis=2).min(1)\n    runs = {}\n    for d in sorted(wd.glob("[ABC]_*")):\n        res = read_run(d, n)\n        if res is not None:\n            if res["x"] is not None:\n                dev = np.abs(res["x"] - prot).max()\n                if dev > 1e-3:\n                    raise RuntimeError(f"{d.name}: atom order or geometry differs from the input ({dev:.4f} A)")\n            runs[d.name] = res\n    lines = [f"runs read: {\', \'.join(sorted(runs))}", f"protein atoms: {n}; farthest from the glycine: {r.max():.1f} A"]\n    table = []\n\n    def diff(a, b, what):\n        if a not in runs or b not in runs:\n            return None\n        if what == "q":\n            return np.abs(runs[a]["q"] - runs[b]["q"])\n        if runs[a]["g"] is None or runs[b]["g"] is None:\n            return None\n        return np.linalg.norm(runs[a]["g"] - runs[b]["g"], axis=1)\n\n    for what, unit in (("q", "e"), ("g", "kcal/mol/A")):\n        lines.append(f"\\n### {\'atomic charges\' if what == \'q\' else \'forces\'} ({unit}): median / 95th percentile / "\n                     f"largest per distance bin")\n        pairs = [("A-B (whole effect)", "A_vac", "B_vac"), ("C-B (point charges)", "C_vac", "B_vac"),\n                 ("A-C (beyond point charges)", "A_vac", "C_vac"),\n                 ("noise A-A", "A_vac", "A_r2"), ("noise B-B", "B_vac", "B_r2"), ("noise C-C", "C_vac", "C_r2"),\n                 ("GPU-CPU A", "A_vac", "A_cpu"),\n                 ("A-C, CPU", "A_cpu", "C_cpu"), ("A-B, CPU", "A_cpu", "B_cpu"),\n                 ("A-C, CUTOFF=15", "A_c15", "C_c15"), ("A-C, CUTOFF=25", "A_c25", "C_c25"),\n                 ("A-B, CUTOFF=25", "A_c25", "B_c25"),\n                 ("A-B, water (COSMO)", "A_eps", "B_eps"), ("A-C, water (COSMO)", "A_eps", "C_eps"),\n                 ("A-B, conventional SCF", "A_conv", "B_conv")]\n        if what == "q":\n            pairs.append(("A-C, conventional SCF (C: point potentials)", "A_conv", "C_conv"))\n        hdr = "  " + " " * 44 + "".join(f"{lo:>4d}-{hi:<4d}" + " " * 14 for lo, hi in zip(BINS, BINS[1:]))\n        lines.append("  bins (A from the glycine): " + ", ".join(f"{lo}-{hi}" for lo, hi in zip(BINS, BINS[1:])))\n        for label, a, b in pairs:\n            v = diff(a, b, what)\n            if v is None:\n                continue\n            bs = binned(r, v)\n            table += [{"quantity": what, "pair": label, "lo": lo, "hi": hi, "n": k, "median": med, "p95": p95,\n                       "max": mx} for lo, hi, k, med, p95, mx in bs]\n            lines.append(f"  {label:44s} " + " | ".join(f"{med:.1e}/{p95:.1e}/{mx:.1e}" for _, _, _, med, p95, mx in bs))\n    with open(od / "binned.csv", "w", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=["quantity", "pair", "lo", "hi", "n", "median", "p95", "max"])\n        w.writeheader()\n        w.writerows(table)\n    # residues: largest |A-C| charge change per residue against its distance; the aromatic path singled out\n    if "A_vac" in runs and "C_vac" in runs:\n        dq = np.abs(runs["A_vac"]["q"] - runs["C_vac"]["q"])\n        noise = np.abs(runs["A_vac"]["q"] - runs["A_r2"]["q"]) if "A_r2" in runs else None\n        res = {}\n        for i, (rn, rid, an, kind) in enumerate(meta["meta"]):\n            k = f"{rn}{rid}"\n            res.setdefault(k, []).append(i)\n        path = {"PHE92", "TRP223", "PHE221", "TYR203", "HIS201", "HIS197", "TYR173", "PHE171"}\n        lines.append("\\n### residues: largest |A-C| charge change (e) and distance of the residue from the glycine (A)")\n        rows = sorted(((dq[ix].max(), r[ix].min(), k) for k, ix in res.items()), key=lambda t: t[1])\n        for lo, hi in zip(BINS, BINS[1:]):\n            sel = [t for t in rows if lo <= t[1] < hi]\n            if not sel:\n                continue\n            top = max(sel)\n            pth = [f"{k}:{m:.1e}" for m, d_, k in sel if k in path]\n            lines.append(f"  {lo:>2d}-{hi:<2d} A: {len(sel):3d} residues, median {np.median([t[0] for t in sel]):.1e}, "\n                         f"largest {top[0]:.1e} ({top[2]})" + (f"; path residues {\', \'.join(pth)}" if pth else ""))\n        if noise is not None:\n            lines.append(f"  noise (A twice): median {np.median(noise):.1e}, 99th percentile {np.percentile(noise, 99):.1e},"\n                         f" largest {noise.max():.1e}")\n        try:\n            import matplotlib\n            matplotlib.use("Agg")\n            import matplotlib.pyplot as plt\n            fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))\n            for ax, what, unit in ((axs[0], "q", "e"), (axs[1], "g", "kcal/mol/Å")):\n                for label, a, b, c in (("A−B", "A_vac", "B_vac", "#1f5fa8"), ("C−B", "C_vac", "B_vac", "#9a9a9a"),\n                                       ("A−C", "A_vac", "C_vac", "#c2452d"), ("noise A−A", "A_vac", "A_r2", "#2b8a3e")):\n                    v = diff(a, b, what)\n                    if v is not None:\n                        ax.scatter(r, np.maximum(v, 1e-9), s=2, alpha=0.35, color=c, label=label)\n                ax.set_yscale("log")\n                ax.set_xlabel("distance from the glycine (Å)")\n                ax.set_ylabel(f"|change| ({unit})")\n                ax.legend(markerscale=5, fontsize=8)\n            fig.tight_layout()\n            fig.savefig(od / "decay.png", dpi=150)\n        except Exception as e:      # the figure is optional\n            lines.append(f"(no figure: {e})")\n    (od / "summary.txt").write_text("\\n".join(lines) + "\\n")\n    print("\\n".join(lines))\n\n\nif __name__ == "__main__":\n    cmd = sys.argv[1]\n    if cmd == "decks":\n        write_decks(Path(sys.argv[2]), Path(sys.argv[3]), int(sys.argv[4]))\n    elif cmd == "mmcharges":\n        print(write_mmcharges(Path(sys.argv[2])))\n    elif cmd == "analyse":\n        analyse(Path(sys.argv[2]), Path(sys.argv[3]))\n')
Path('/content/run_t1.py').write_text('"""Run test-1 MOPAC jobs: python run_t1.py MOPAC WORK_DIR JOB[,JOB...] [PARALLEL] [TIMEOUT_S]\n\nEach job directory holds job.mop and env.json (extra environment); finished jobs (job.aux present and a\nstatus.json) are skipped. status.json: wall time, exit code, SCF status lines.\n"""\nimport json\nimport os\nimport re\nimport subprocess\nimport sys\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\n\ndef run_job(mopac, d, timeout):\n    st = d / "status.json"\n    if st.exists() and (d / "job.aux").exists():\n        return json.loads(st.read_text())\n    env = dict(os.environ)\n    if (d / "env.json").exists():\n        env.update(json.loads((d / "env.json").read_text()))\n    t0 = time.time()\n    try:\n        p = subprocess.run([mopac, "job.mop"], cwd=d, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                           text=True, timeout=timeout)\n        code, log = p.returncode, p.stdout\n    except subprocess.TimeoutExpired:\n        code, log = "timeout", ""\n    out = (d / "job.out").read_text(errors="ignore") if (d / "job.out").exists() else ""\n    res = {"job": d.name, "seconds": round(time.time() - t0, 1), "exit": code,\n           "heat": (re.findall(r"FINAL HEAT OF FORMATION =\\s+(-?\\d+\\.\\d+)", out) or [None])[-1],\n           "scf": re.findall(r"\\[MOZYME GPU SCF\\] status=\\S+.*", log + out)[-3:],\n           "errors": [ln.strip() for ln in out.splitlines() if "ERROR" in ln.upper()][:5]}\n    st.write_text(json.dumps(res, indent=1))\n    return res\n\n\nif __name__ == "__main__":\n    mopac, wd = sys.argv[1], Path(sys.argv[2])\n    jobs = [wd / j for j in sys.argv[3].split(",")]\n    par = int(sys.argv[4]) if len(sys.argv) > 4 else 1\n    timeout = float(sys.argv[5]) if len(sys.argv) > 5 else None\n    with ThreadPoolExecutor(par) as ex:\n        for r in ex.map(lambda d: run_job(mopac, d, timeout), jobs):\n            print(json.dumps(r), flush=True)\n')

Path('/content/1PB7_atoms.csv').write_bytes(gzip.decompress(base64.b64decode('...')))
WD = OUT / 'runs'
if not (WD / 'meta.json').exists():
    run(f'{sys.executable} /content/t1_tools.py decks /content/1PB7_atoms.csv {WD} 2')
run(f'{sys.executable} /content/run_t1.py {MOPAC} {WD} D')
run(f'{sys.executable} /content/t1_tools.py mmcharges {WD}')

## 5. Corridas

Las tres corridas en CPU arrancan en segundo plano (una por núcleo) y las de GPU van una tras otra.

In [ ]:
cpu_log = OUT / 'cpu_jobs.log'
cpu = subprocess.Popen(f'{sys.executable} /content/run_t1.py {MOPAC} {WD} A_cpu,B_cpu,C_cpu 3 > {cpu_log} 2>&1',
                       shell=True, executable='/bin/bash')
gpu_jobs = ['A_vac', 'B_vac', 'C_vac', 'A_r2', 'B_r2', 'C_r2', 'A_c15', 'B_c15', 'C_c15', 'A_c25', 'B_c25', 'C_c25',
            'A_eps', 'B_eps', 'C_eps']
run(f'{sys.executable} /content/run_t1.py {MOPAC} {WD} {",".join(gpu_jobs)} 1')
run(f'{sys.executable} /content/t1_tools.py analyse {WD} {OUT / "analysis"}')

## 6. SCF convencional (sin orbitales localizados)

Control de MOZYME: el mismo A − B con la matriz completa. Hasta `CONV_TIMEOUT` segundos por corrida; si no termina, el
resto de los resultados vale igual.

In [ ]:
run(f'{sys.executable} /content/run_t1.py {MOPAC} {WD} A_conv,B_conv,C_conv 1 {CONV_TIMEOUT}', check=False)
print('esperando las corridas en CPU...', flush=True)
cpu.wait()
run(f'cat {cpu_log}', check=False)
run(f'{sys.executable} /content/t1_tools.py analyse {WD} {OUT / "analysis"}')

## 7. Figura y descarga

In [ ]:
from IPython.display import Image, display
if (OUT / 'analysis' / 'decay.png').exists():
    display(Image(str(OUT / 'analysis' / 'decay.png')))
zipbase = f'/content/{RESULTS_NAME}'
shutil.make_archive(zipbase, 'zip', OUT.parent, OUT.name)
print(f'{zipbase}.zip', os.path.getsize(f'{zipbase}.zip') // 1024, 'kB')
try:
    from google.colab import files
    files.download(f'{zipbase}.zip')
except Exception as e:
    print('descárguelo a mano:', e)